# 03 — Silver Cleaning

## Background
This notebook implements the profiling decisions (`02_data_profiling`) 
to produce three clean, typed, and joined tables, ready for EDA 
and Gold feature engineering.

## Decisions Implemented (Based on Profiling)

1. **No columns removed.** Profiling confirmed 0 columns 
   that were 100% NULL and 0 constant columns (cardinality = 1) among the 435 columns 
   audited.
2. **No imputation.** NULL values are informative—tree-based models 
   treat them natively as a modality.
3. **No encoding.** One-hot, frequency, and target encoding are excluded 
   from Silver: they belong to the ML pipeline (fold-safe), not to data cleaning.
4. **A single sentinel mapping**: `id_34 = “match_status:-1”` → NULL. 
   Decision validated by profiling (3 lines, system error code). 
   The other detected sentinels (`NotFound`, `Unknown`) are 
   dominant business categories that are retained.
5. **Snake_case renaming** + canonical typing via PySpark metaprogramming 
   (loop over `df.columns`, no hard-coding).
6. **Defensive deduplication** on `transaction_id` (expected 0 duplicates, 
   safeguard enabled).
7. **Join** `transactions LEFT JOIN identity` on `transaction_id` 
   (24.4% device coverage, 0 orphans).
## Deliverables

| Table | Rows | Columns |
|---|---|---|
| `fraud.silver.transactions_clean` | 590,540 | 394 |
| `fraud.silver.identity_clean` | 144,233 | 41 |
| `fraud.silver.transactions_identity_joined` | 590,540 | 434 |

## What Silver Doesn't Do (and Why)

| Operation | Where it occurs |
|---|---|
| Feature selection | Step 6 (`06_gold_selection`) |
| Feature engineering (UID, velocity) | Step 5 (`05_gold_features`) |
| Encoding (one-hot, frequency, target) | Step 6, fold-safe |
| NULL imputation | Nowhere — LightGBM/XGBoost handle this natively |
| Feature-target statistical analysis | Step 4 (`04_eda`) |
| Temporal split | Step 6 |

**Guiding Principle**: Silver = clean data warehouse. No selection or 
ML processing. The analysis informs the Gold, not the other way around.

In [0]:
%sql
-- Creating silver schema and volume
-- The SILVER schema is the “Cleaned & Refined” layer.
-- Objective: To store data after it has been cleaned, typed, and standardized.
-- This layer serves as the Single Source of Truth (SSoT) for feature engineering.
CREATE SCHEMA IF NOT EXISTS fraud.silver
COMMENT 'Cleaned dataset: typed, deduplicated, and ready for EDA';


In [0]:
#Helpers
from pyspark.sql import functions as F
import re

#Converts a CamelCase name to snake_case.
def to_snake(name: str) -> str:
    """TransactionID -> transaction_id, ProductCD -> product_cd"""
    # Explicit exceptions for known columns
    if name == "TransactionID":
        return "transaction_id"
    if name == "isFraud":
        return "is_fraud"
    if name == "TransactionDT":
        return "transaction_dt"
    if name == "TransactionAmt":
        return "transaction_amt"
    if name == "ProductCD":
        return "product_cd"
    # General case: underscore between lowercase and uppercase letters
    s = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', '_', name)
    return s.lower()

#Applies type casting and renames each column, then returns a transformed DataFrame.
def cast_transactions(df):
    exprs = []
    for c in df.columns:
        new = to_snake(c)
        if c == "TransactionID":
            exprs.append(F.col(f"`{c}`").cast("bigint").alias(new))
        elif c == "isFraud":
            exprs.append(F.col(f"`{c}`").cast("int").alias(new))
        elif c == "TransactionDT":
            exprs.append(F.col(f"`{c}`").cast("bigint").alias(new))
        elif c == "TransactionAmt":
            exprs.append(F.col(f"`{c}`").cast("double").alias(new))
        elif c[0] in ("V", "C", "D") and c[1:].isdigit():
            exprs.append(F.col(f"`{c}`").cast("double").alias(new))
        elif c[0] == "M" and c[1:].isdigit():
            exprs.append(F.col(f"`{c}`").cast("string").alias(new))
        elif c in ("card1", "card2", "card3", "card5"):
            exprs.append(F.col(f"`{c}`").cast("int").alias(new))
        elif c in ("card4", "card6", "addr1", "addr2"):
            exprs.append(F.col(f"`{c}`").cast("string").alias(new))
        elif c.startswith("dist"):
            exprs.append(F.col(f"`{c}`").cast("double").alias(new))
        elif c in ("ProductCD", "P_emaildomain", "R_emaildomain"):
            exprs.append(F.lower(F.col(f"`{c}`").cast("string")).alias(new))
        else:
            exprs.append(F.col(f"`{c}`").alias(new))
    return df.select(*exprs)

# The same logic as `cast_transactions`, but adapted to the specific characteristics of the identity table.
# id_* columns of type STRING 
STRING_ID_COLUMNS = {
    "id_12", "id_15", "id_16", "id_23", "id_27",
    "id_28", "id_29", "id_30", "id_31", "id_33", "id_34",
    "id_35", "id_36", "id_37", "id_38",
}

def cast_identity(df):
    exprs = []
    for c in df.columns:
        if c == "TransactionID":
            exprs.append(F.col(f"`{c}`").cast("bigint").alias("transaction_id"))
        elif c in ("DeviceType", "DeviceInfo"):
            exprs.append(F.lower(F.col(f"`{c}`").cast("string")).alias(to_snake(c)))
        elif c in STRING_ID_COLUMNS:
            exprs.append(F.col(f"`{c}`").cast("string").alias(c))
        elif c.startswith("id_"):
            exprs.append(F.col(f"`{c}`").cast("double").alias(c))
        else:
            exprs.append(F.col(f"`{c}`").alias(c.lower()))
    return df.select(*exprs)

In [0]:
# Applies the transformation defined in cell 2, then writes the result to the Delta table.
df_tx = spark.table("fraud.bronze.transactions_raw")
df_tx_clean = cast_transactions(df_tx).dropDuplicates(["transaction_id"])

df_tx_clean.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud.silver.transactions_clean")

print(f"transactions_clean : {df_tx_clean.count():,} rows, {len(df_tx_clean.columns)} columns")

transactions_clean : 590,540 rows, 394 columns


In [0]:
# Type casting, renaming, and deduplication of the identity table+ sentinel
spark.sql("DROP TABLE IF EXISTS fraud.silver.identity_clean")
df_id = spark.table("fraud.bronze.identity_raw")
df_id_clean = cast_identity(df_id).dropDuplicates(["transaction_id"])

# Only one sentinel validated by profiling: id_34 = “match_status:-1” -> NULL
df_id_clean = df_id_clean.withColumn(
    "id_34",
    F.when(F.col("id_34") == "match_status:-1", F.lit(None)).otherwise(F.col("id_34"))
)

df_id_clean.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud.silver.identity_clean")

print(f"identity_clean : {df_id_clean.count():,} rows, {len(df_id_clean.columns)} columns")

identity_clean : 144,233 rows, 41 columns


In [0]:
# Identity table transformations checking
df_check = spark.table("fraud.silver.identity_clean")
print(f"identity_clean : {df_check.count():,} lignes, {len(df_check.columns)} colonnes")
df_check.printSchema()

identity_clean : 144,233 lignes, 41 colonnes
root
 |-- transaction_id: long (nullable = true)
 |-- id_01: double (nullable = true)
 |-- id_02: double (nullable = true)
 |-- id_03: double (nullable = true)
 |-- id_04: double (nullable = true)
 |-- id_05: double (nullable = true)
 |-- id_06: double (nullable = true)
 |-- id_07: double (nullable = true)
 |-- id_08: double (nullable = true)
 |-- id_09: double (nullable = true)
 |-- id_10: double (nullable = true)
 |-- id_11: double (nullable = true)
 |-- id_12: string (nullable = true)
 |-- id_13: double (nullable = true)
 |-- id_14: double (nullable = true)
 |-- id_15: string (nullable = true)
 |-- id_16: string (nullable = true)
 |-- id_17: double (nullable = true)
 |-- id_18: double (nullable = true)
 |-- id_19: double (nullable = true)
 |-- id_20: double (nullable = true)
 |-- id_21: double (nullable = true)
 |-- id_22: double (nullable = true)
 |-- id_23: string (nullable = true)
 |-- id_24: double (nullable = true)
 |-- id_25: double

In [0]:
# Join: Transactions + Identity
df_tx_clean = spark.table("fraud.silver.transactions_clean")
df_id_clean = spark.table("fraud.silver.identity_clean")

df_joined = df_tx_clean.join(df_id_clean, on="transaction_id", how="left")

df_joined.write.format("delta").mode("overwrite") \
    .saveAsTable("fraud.silver.transactions_identity_joined")

print(f"transactions_identity_joined : {df_joined.count():,} rows, {len(df_joined.columns)} columns")

transactions_identity_joined : 590,540 rows, 434 columns


In [0]:
# Final verification
for tbl in ["transactions_clean", "identity_clean", "transactions_identity_joined"]:
    n = spark.table(f"fraud.silver.{tbl}").count()
    print(f"  fraud.silver.{tbl:<30} {n:>10,} rows")

# Consistency Checks
assert spark.table("fraud.silver.transactions_clean").count() == 590_540
assert spark.table("fraud.silver.identity_clean").count() == 144_233
assert spark.table("fraud.silver.transactions_identity_joined").count() == 590_540

# Verify that sentinel id_34 has been properly mapped
n_sentinel = spark.table("fraud.silver.identity_clean") \
    .filter(F.col("id_34") == "match_status:-1").count()
assert n_sentinel == 0, f"Sentinel non mapped : {n_sentinel} rows"

print("Silver validated")

  fraud.silver.transactions_clean                590,540 rows
  fraud.silver.identity_clean                    144,233 rows
  fraud.silver.transactions_identity_joined      590,540 rows
Silver validated
